# 🌙 12. Sub-Pixel Enhanced Correlation Coefficient (ECC) Maximization

**Mission Context**: Fine-scale photometric sub-pixel refinement ($< 0.1\text{ px}$) invariant to linear radiometric contrast variations.  
**Objectives**:
- Optimize alignment using iterative ECC gradient ascent.
- Measure ECC correlation coefficient, pixel improvement %, and residual alignment error.
- Export `ecc_registered.png` and `ecc_metrics.json`.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
import json

sys.path.append(str(Path.cwd().parent))
from lunar_core.config import load_config
from lunar_core.ecc import ECCRefinementEngine
from lunar_core.synthetic_data import LunarSyntheticGenerator

config = load_config()
gen = LunarSyntheticGenerator(size=(512, 512), seed=42)
pair = gen.generate_registered_pair(rotation_deg=8.0, scale=1.02, tx=15.0, ty=-10.0)

ref_img, src_img = pair["reference_image"], pair["source_image"]
H_gt = pair["homography_ground_truth"]

# Add slight initial perturbation to simulate coarse feature registration error
H_perturbed = H_gt.copy()
H_perturbed[0, 2] += 1.5
H_perturbed[1, 2] -= 1.2
coarse_warped = cv2.warpPerspective(src_img, H_perturbed, (512, 512))

ecc_engine = ECCRefinementEngine(max_iterations=120, termination_eps=1e-6)
ecc_res = ecc_engine.refine(ref_img, coarse_warped)

print("--- ECC Sub-Pixel Refinement Results ---")
print(f"Convergence Success: {ecc_res['success']}")
print(f"Final ECC Score: {ecc_res['ecc_score']}")
print(f"Initial Error: {ecc_res['initial_alignment_error']} -> Final Error: {ecc_res['final_alignment_error']}")
print(f"Pixel Error Improvement: {ecc_res['pixel_improvement_pct']}%")


In [ ]:
# Visualize ECC Refinement
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(coarse_warped, cmap='gray')
axes[0].set_title(f"Coarse Registered Frame (Err: {ecc_res['initial_alignment_error']})", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(ecc_res["refined_image"], cmap='gray')
axes[1].set_title(f"ECC Refined Frame (Err: {ecc_res['final_alignment_error']}, +{ecc_res['pixel_improvement_pct']}%)", fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
os.makedirs("outputs/visualizations", exist_ok=True)
plt.savefig("outputs/visualizations/12_ecc_refinement.png", dpi=300)
plt.show()


In [ ]:
# Export ECC outputs
os.makedirs("outputs/registered", exist_ok=True)
cv2.imwrite("outputs/registered/ecc_registered.png", ecc_res["refined_image"])

metrics = {
    "success": ecc_res["success"],
    "ecc_score": ecc_res["ecc_score"],
    "initial_alignment_error": ecc_res["initial_alignment_error"],
    "final_alignment_error": ecc_res["final_alignment_error"],
    "pixel_improvement_pct": ecc_res["pixel_improvement_pct"]
}
with open("outputs/registered/ecc_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Exported outputs/registered/ecc_registered.png and ecc_metrics.json")
